# Llama-3.2-3B IT Ticket Classifier — v2 Fixed

## All fixes applied (from paper + bug analysis):
- ✅ `early_stopping`: per-stage comparison (`stage_best_f1 = 0.0` each stage), `patience=2`
- ✅ `scheduler`: single global scheduler across all stages (not rebuilt per-stage)
- ✅ `lora_alpha = 2 * r` (paper Table 1: r=8, alpha=16)
- ✅ `lora_dropout = 0.05` (paper Table 1)
- ✅ `max_grad_norm = 1.0` (paper Table 1)
- ✅ `optimizer = AdamW8bit` (paper Table 1)
- ✅ `scheduler = Linear, no warmup` (paper Table 1)
- ✅ `head = single Linear` 3072→10 (paper approach, simpler)
- ✅ `STAGE_SIZE = 3388` → uses 100% of training data (7×3388=23716)
- ✅ `GRAD_ACCUM = 16` → effective batch = 32
- ✅ `MAX_EPOCHS = 3` — overfitting observed at ep3, no increase
- ✅ HPT: r=8 (paper best) vs r=16 vs r=32

In [ ]:
import json
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')

old_path = Path("/content/drive/MyDrive/Colab Notebooks/llama3_2_3b.ipynb")
new_path = Path("/content/drive/MyDrive/Colab Notebooks/llama3_2_3bl.ipynb")

nb = json.loads(old_path.read_text(encoding="utf-8"))

# امسح widgets من metadata الرئيسية
nb.get("metadata", {}).pop("widgets", None)

# امسح outputs كمان احتياطيًا
for cell in nb.get("cells", []):
    cell["outputs"] = []
    cell["execution_count"] = None

new_path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

# تحقق
check = json.loads(new_path.read_text(encoding="utf-8"))
print("widgets exists?", "widgets" in check.get("metadata", {}))
print("Saved fixed file:", new_path)

files.download(str(new_path))

## Cell 1 — Install Libraries

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q "transformers>=4.45.0,<5.0.0"
!pip install -q datasets scikit-learn matplotlib seaborn
print('✅ Installation done')

## Cell 2 — Imports

In [ ]:
import os, gc, json, math, random, shutil, torch, torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding, get_linear_schedule_with_warmup
from unsloth import FastLanguageModel
from peft import PeftModel
import bitsandbytes as bnb

print('✅ All imports done')
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB'
      if torch.cuda.is_available() else '')

## Cell 3 — Configuration

In [ ]:
# ══════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════
DATA_PATH       = '/content/drive/MyDrive/IT Support Ticket Data.csv'
CHECKPOINT_BASE = '/content/drive/MyDrive/llama32-ticket-cls-v2'

# ══════════════════════════════════════════════════
# MODEL
# Paper evidence: Base (non-instruct) = F1=0.860
# Instruct = F1=0.601 (Table 2, Yousefiramandi & Cooney 2025)
# ══════════════════════════════════════════════════
MODEL_NAME  = 'unsloth/Llama-3.2-3B-bnb-4bit'
MAX_SEQ_LEN = 256   # covers 99.4% of IT tickets

# ══════════════════════════════════════════════════
# DATA SPLIT  (same as Classical ML for fair comparison)
# ══════════════════════════════════════════════════
RANDOM_STATE = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.5

# ══════════════════════════════════════════════════
# STAGED TRAINING
# FIX: STAGE_SIZE = 3388 → 7×3388=23716 ≈ 100% of train data
# Previously 7×3000=21000 (89%) — 2720 samples were unused
# ══════════════════════════════════════════════════
STAGE_SIZE = 3000
NUM_STAGES = 7

# ══════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# Source: paper Table 1 + Unsloth docs
# ══════════════════════════════════════════════════
BATCH_SIZE              = 2
EVAL_BATCH_SIZE         = 8
GRAD_ACCUM              = 16    # FIX: was 8 → effective batch = 32 (more stable)
MAX_EPOCHS              = 3     # overfitting at ep3, no increase justified
EARLY_STOPPING_PATIENCE = 2     # FIX: was 1. Paper uses patience=2
WEIGHT_DECAY            = 0.01  # paper Table 1
MAX_GRAD_NORM           = 1.0   # FIX: was 0.3. Paper Table 1 uses 1.0

# ══════════════════════════════════════════════════
# LORA TARGET MODULES
# All 7 major linear layers (paper + Unsloth docs)
# ══════════════════════════════════════════════════
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# ══════════════════════════════════════════════════
# HPT CONFIGS
# Key fixes vs previous version:
#   alpha = 2*r  (paper Table 1: r=8 alpha=16 → alpha=2r)
#   dropout = 0.05 (paper Table 1)
#   lr = 2e-4 (paper + Unsloth docs)
# ══════════════════════════════════════════════════
LORA_CONFIGS = [
    # Paper best: r=8, alpha=16 (=2r), F1=0.860
    {'name': 'LoRA_r8',  'r': 8,  'alpha': 16, 'lr': 2e-4, 'dropout': 0.05},
    # Second: r=16, alpha=32 (=2r)
    {'name': 'LoRA_r16', 'r': 16, 'alpha': 32, 'lr': 2e-4, 'dropout': 0.05},
    # Third: r=32, alpha=64 (=2r) — higher capacity, more VRAM
    {'name': 'LoRA_r32', 'r': 32, 'alpha': 64, 'lr': 2e-4, 'dropout': 0.05},
]

# ══════════════════════════════════════════════════
# CLASSES
# ══════════════════════════════════════════════════
DEPARTMENTS = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support',
]
LABEL2ID = {d: i for i, d in enumerate(DEPARTMENTS)}
ID2LABEL = {i: d for d, i in LABEL2ID.items()}
NUM_CLASSES = len(DEPARTMENTS)

print('✅ Config loaded')
print(f'   Model       : {MODEL_NAME}')
print(f'   Stage size  : {STAGE_SIZE} × {NUM_STAGES} = {STAGE_SIZE*NUM_STAGES} samples')
print(f'   Eff. batch  : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'   Patience    : {EARLY_STOPPING_PATIENCE} (per-stage)')
print(f'   Grad norm   : {MAX_GRAD_NORM}')
print()
print('HPT Configs:')
for c in LORA_CONFIGS:
    print(f'  {c["name"]}: r={c["r"]} alpha={c["alpha"]} (={c["alpha"]//c["r"]}r) '
          f'dropout={c["dropout"]} lr={c["lr"]}')

## Cell 4 — Mount Drive & Load Data

In [ ]:
drive.mount('/content/drive')
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

df = pd.read_csv(DATA_PATH, index_col=0)
df = df.dropna(subset=['Body']).reset_index(drop=True)
df = df[['Body', 'Department']]

print(f'Total samples : {len(df)}')
print(f'Departments   : {df["Department"].nunique()}')
print()
print(df['Department'].value_counts())

## Cell 5 — Data Split + Stratified Stages

In [ ]:
# Same split as Classical ML for fair comparison
train_df, temp_df = train_test_split(
    df, test_size=TEST_SIZE,
    stratify=df['Department'], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=VAL_SIZE,
    stratify=temp_df['Department'], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)


def make_stratified_stages(df, stage_size, num_stages, random_state=42):
    """
    Non-overlapping stratified stages.
    FIX: stage_size=3388 → 7×3388=23716 uses ~100% of training data.
    Previously 7×3000=21000 wasted 2720 samples.
    """
    total_needed = stage_size * num_stages
    available = len(df)
    if total_needed > available:
        # Adjust stage_size to fit available data
        stage_size = available // num_stages
        total_needed = stage_size * num_stages
        print(f'  Adjusted stage_size to {stage_size} (total={total_needed})')

    sampled, _ = train_test_split(
        df, train_size=total_needed,
        stratify=df['Department'], random_state=random_state
    )
    sampled = sampled.reset_index(drop=True)

    stages = []
    remaining = sampled.copy()
    for i in range(num_stages - 1):
        stage, remaining = train_test_split(
            remaining, train_size=stage_size,
            stratify=remaining['Department'], random_state=random_state + i
        )
        stages.append(stage.reset_index(drop=True))
    stages.append(remaining.reset_index(drop=True))
    return stages


stages = make_stratified_stages(train_df, STAGE_SIZE, NUM_STAGES)

print('=' * 60)
print('  DATA SPLIT SUMMARY')
print('=' * 60)
print(f'  Train  : {len(train_df):>6}  (80%)')
print(f'  Val    : {len(val_df):>6}  (10%)')
print(f'  Test   : {len(test_df):>6}  (10%)')
print(f'  Stages : {NUM_STAGES} × {len(stages[0])} = {sum(len(s) for s in stages)} '
      f'({sum(len(s) for s in stages)/len(train_df)*100:.1f}% of train)')
print()
for i, s in enumerate(stages):
    counts = s['Department'].value_counts()
    print(f'  Stage {i+1}: {len(s)} samples | '
          f'min={counts.min()} max={counts.max()} (std={counts.std():.1f})')

## Cell 6 — Tokenizer + Dataset + Dynamic Padding

In [ ]:
# Load tokenizer once — shared across all experiments
_, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Paper Table 1: add_prefix_space=True; padding_side=right
tokenizer.padding_side = 'right'


class TicketDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts     = dataframe['Body'].astype(str).tolist()
        self.labels    = [LABEL2ID[d] for d in dataframe['Department'].tolist()]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         self.labels[idx],
        }


# Dynamic padding — pad only to longest in each batch
# Paper Table 1: Dynamic (DataCollatorWithPadding)
base_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='longest')


class TicketCollator:
    """Wraps DataCollatorWithPadding to handle the labels field."""
    def __init__(self, collator):
        self.collator = collator

    def __call__(self, features):
        labels = [f.pop('labels') for f in features]
        batch  = self.collator(features)
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch


collator = TicketCollator(base_collator)

val_loader = DataLoader(
    TicketDataset(val_df, tokenizer, MAX_SEQ_LEN),
    batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator,
)
test_loader = DataLoader(
    TicketDataset(test_df, tokenizer, MAX_SEQ_LEN),
    batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator,
)

print('✅ Tokenizer + Dynamic padding ready')
print(f'   Val batches  : {len(val_loader)}')
print(f'   Test batches : {len(test_loader)}')

## Cell 7 — Model: Llama-3.2-3B + rsLoRA + Simple Linear Head

In [ ]:
class LlamaTicketClassifier(nn.Module):
    """
    Embedding-based classification (Approach 1 from paper).

    Key design choices from paper (Table 1 + Section 2.1):
    - Last token pooling (decoder-only LLM: last token sees full context)
    - Single Linear head: hidden_size → num_classes  (NO hidden layer)
      Paper uses simple linear layer, not 2-layer MLP
    - head in float32 to avoid NaN loss with 4-bit backbone
    - rsLoRA enabled (use_rslora=True)
    - All 7 linear layers targeted
    """

    def __init__(self, model_name, num_classes, lora_cfg,
                 adapter_path=None, is_trainable=True):
        super().__init__()
        self.lora_cfg = lora_cfg

        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
        )

        if adapter_path is None:
            # Fresh training
            self.backbone = FastLanguageModel.get_peft_model(
                base_model,
                r                        = lora_cfg['r'],
                lora_alpha               = lora_cfg['alpha'],   # = 2*r (paper)
                target_modules           = LORA_TARGET_MODULES,
                lora_dropout             = lora_cfg['dropout'], # 0.05 (paper)
                bias                     = 'none',
                use_rslora               = True,
                use_gradient_checkpointing = 'unsloth',
                random_state             = RANDOM_STATE,
            )
            print('Loaded fresh rsLoRA adapters for training.')
        else:
            self.backbone = PeftModel.from_pretrained(
                base_model, adapter_path, is_trainable=is_trainable,
            )
            print(f'Loaded adapters from: {adapter_path}')

        self.backbone.print_trainable_parameters()

        hidden_size = self.backbone.config.hidden_size  # 3072 for Llama-3.2-3B

        # ── Single Linear head (paper: simple linear layer)
        # FIX: was 2-layer MLP — paper uses single linear, same F1=0.860
        self.classifier = nn.Linear(hidden_size, num_classes).float()
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        self.classifier = self.classifier.to(device=self.backbone.device)

        self.loss_fn = nn.CrossEntropyLoss()

        print()
        print('✅ LlamaTicketClassifier ready')
        print(f'   Hidden size : {hidden_size}')
        print(f'   Head        : {hidden_size} → {num_classes}  (single linear)')
        print(f'   rsLoRA      : enabled | alpha={lora_cfg["alpha"]} = {lora_cfg["alpha"]//lora_cfg["r"]}r')
        print(f'   Pooling     : last non-padding token')
        print(f'   Dropout     : {lora_cfg["dropout"]} (LoRA)')

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # Last non-padding token
        last_hidden  = outputs.hidden_states[-1]
        seq_lengths  = attention_mask.sum(dim=1) - 1
        batch_size   = input_ids.shape[0]
        pooled = last_hidden[
            torch.arange(batch_size, device=input_ids.device),
            seq_lengths,
        ].float()   # cast to float32 for head

        logits = self.classifier(pooled)
        loss   = self.loss_fn(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}


def build_model(lora_cfg, adapter_path=None, is_trainable=True):
    return LlamaTicketClassifier(
        model_name   = MODEL_NAME,
        num_classes  = NUM_CLASSES,
        lora_cfg     = lora_cfg,
        adapter_path = adapter_path,
        is_trainable = is_trainable,
    )


print('✅ Model class defined')

## Cell 8 — Evaluation Function

In [ ]:
def evaluate(model, loader, split_name='Val', show_plots=True, save_dir=None):
    model.eval()
    device     = next(model.parameters()).device
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            out   = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
            )
            all_preds.extend(out['logits'].argmax(dim=-1).cpu().numpy())
            all_labels.extend(batch['labels'].numpy())

    acc    = accuracy_score(all_labels, all_preds)
    f1m    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    f1w    = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_pc  = f1_score(all_labels, all_preds, average=None,       zero_division=0)
    cm     = confusion_matrix(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds, target_names=DEPARTMENTS,
        zero_division=0, output_dict=True
    )

    print('=' * 65)
    print(f'  {split_name}')
    print('=' * 65)
    print(f'  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  F1 Macro    : {f1m:.4f}')
    print(f'  F1 Weighted : {f1w:.4f}')
    print()
    print(classification_report(
        all_labels, all_preds, target_names=DEPARTMENTS, zero_division=0
    ))

    if show_plots:
        fig = plt.figure(figsize=(18, 7))
        gs  = gridspec.GridSpec(1, 2, width_ratios=[1.8, 1])

        ax1 = fig.add_subplot(gs[0])
        short = [d[:18] for d in DEPARTMENTS]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=short, yticklabels=short, ax=ax1)
        ax1.set_title(f'Confusion Matrix — {split_name}', fontsize=13)
        ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(ax1.get_yticklabels(), rotation=0,  fontsize=8)

        ax2 = fig.add_subplot(gs[1])
        colors = ['#2ecc71' if f >= 0.6 else '#e74c3c' if f < 0.4 else '#f39c12'
                  for f in f1_pc]
        ax2.barh(range(NUM_CLASSES), f1_pc, color=colors)
        ax2.set_yticks(range(NUM_CLASSES))
        ax2.set_yticklabels([d[:20] for d in DEPARTMENTS], fontsize=8)
        ax2.set_xlabel('F1 Score')
        ax2.set_title('F1 per Department', fontsize=11)
        ax2.axvline(x=f1m, color='navy', linestyle='--', alpha=0.7,
                    label=f'Macro F1={f1m:.3f}')
        ax2.legend(fontsize=9)
        ax2.set_xlim(0, 1)

        plt.suptitle(f'{split_name} | Acc={acc:.3f} | F1-Macro={f1m:.3f}',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            path = os.path.join(save_dir, f'eval_{split_name.replace(" ","_")}.png')
            plt.savefig(path, dpi=100, bbox_inches='tight')
            print(f'  Plot saved → {path}')
        plt.show()

    model.train()
    return {
        'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w,
        'f1_per_class': f1_pc.tolist(),
        'confusion_matrix': cm.tolist(),
        'per_class_report': report,
    }


print('✅ evaluate() defined')

## Cell 9 — Checkpoint Functions

In [ ]:
def checkpoint_dir(lora_name, tag):
    return os.path.join(CHECKPOINT_BASE, lora_name, tag)

def adapter_dir(lora_name, tag):
    return os.path.join(checkpoint_dir(lora_name, tag), 'lora_adapters')


def save_checkpoint(model, optimizer, scheduler, stage, epoch, lora_name,
                    metrics, best_val_f1, stage_best_f1,
                    patience_counter, train_history,
                    tag=None, is_best=False):
    tag     = tag or f'stage{stage}_ep{epoch}'
    ckpt_dir = checkpoint_dir(lora_name, tag)
    os.makedirs(ckpt_dir, exist_ok=True)

    model.backbone.save_pretrained(adapter_dir(lora_name, tag))
    torch.save(model.classifier.state_dict(),
               os.path.join(ckpt_dir, 'classifier_head.pt'))

    state = {
        'stage': stage, 'epoch': epoch, 'lora_name': lora_name,
        'best_val_f1': best_val_f1, 'stage_best_f1': stage_best_f1,
        'patience_counter': patience_counter, 'metrics': metrics,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'train_history': train_history, 'is_best': is_best, 'tag': tag,
    }
    torch.save(state, os.path.join(ckpt_dir, 'training_state.pt'))

    mj = {
        'stage': stage, 'epoch': epoch, 'lora_config': lora_name,
        'tag': tag, 'is_best': is_best,
        'accuracy':    round(metrics['accuracy'],    6),
        'f1_macro':    round(metrics['f1_macro'],    6),
        'f1_weighted': round(metrics['f1_weighted'], 6),
        'train_loss':  round(metrics.get('train_loss', 0.0), 6),
        'train_acc':   round(metrics.get('train_acc',  0.0), 6),
        'f1_per_class': {DEPARTMENTS[i]: round(v, 4)
                         for i, v in enumerate(metrics['f1_per_class'])},
    }
    with open(os.path.join(ckpt_dir, 'metrics.json'), 'w') as f:
        json.dump(mj, f, indent=2)

    label = '🏆 BEST' if is_best else '💾'
    print(f'  {label} Checkpoint → {ckpt_dir}')
    print(f'     acc={metrics["accuracy"]:.4f} | F1-macro={metrics["f1_macro"]:.4f}')


def load_checkpoint_state(optimizer, scheduler, lora_name, tag='LAST'):
    ckpt_dir = checkpoint_dir(lora_name, tag)
    if not os.path.exists(ckpt_dir):
        print(f'  ⚠️  Not found: {ckpt_dir}')
        return None
    state = torch.load(os.path.join(ckpt_dir, 'training_state.pt'),
                       map_location='cpu')
    optimizer.load_state_dict(state['optimizer'])
    if scheduler and state.get('scheduler'):
        scheduler.load_state_dict(state['scheduler'])
    print(f'  ✅ Loaded state: stage={state["stage"]} ep={state["epoch"]} '
          f'best_f1={state["best_val_f1"]:.4f}')
    return state


def load_classifier_head(model, lora_name, tag='BEST'):
    ckpt_dir = checkpoint_dir(lora_name, tag)
    if not os.path.exists(ckpt_dir):
        print(f'⚠️  Checkpoint not found: {lora_name} @ {tag}')
        return False
    model.classifier.load_state_dict(
        torch.load(os.path.join(ckpt_dir, 'classifier_head.pt'),
                   map_location='cpu')
    )
    print(f'✅ Loaded classifier head from {ckpt_dir}')
    return True


def list_checkpoints(lora_name=None):
    base = os.path.join(CHECKPOINT_BASE, lora_name) if lora_name else CHECKPOINT_BASE
    if not os.path.exists(base):
        print('No checkpoints found.')
        return
    print(f'Checkpoints in {base}:')
    for root, _, files in os.walk(base):
        if 'metrics.json' in files:
            with open(os.path.join(root, 'metrics.json')) as f:
                m = json.load(f)
            flag = ' ← BEST' if m.get('is_best') else ''
            print(f'  {os.path.relpath(root, CHECKPOINT_BASE):45s} '
                  f'acc={m["accuracy"]:.4f} F1={m["f1_macro"]:.4f}{flag}')


print('✅ Checkpoint functions defined')

## Cell 10 — Training Loop

### Key fixes vs v1:
1. **`stage_best_f1 = 0.0`** at start of every stage → per-stage comparison (not global)
2. **`patience=2`** → tolerates 2 non-improving epochs per stage
3. **Single global scheduler** built before the stage loop → correct LR curve
4. **Linear scheduler, no warmup** (paper Table 1)

In [ ]:
def build_stage_loader(stage_df):
    return DataLoader(
        TicketDataset(stage_df, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
    )


def build_global_scheduler(optimizer, stages):
    """
    FIX: Single scheduler for all stages.
    Previously rebuilt per-stage → warmup restart every stage (wrong).
    Paper Table 1: Linear schedule, no warmup.
    """
    # total optimizer steps across all stages × epochs
    steps_per_stage = math.ceil(STAGE_SIZE / (BATCH_SIZE * GRAD_ACCUM))
    total_steps = steps_per_stage * MAX_EPOCHS * NUM_STAGES
    total_steps = max(total_steps, 1)

    # Linear decay, no warmup (paper Table 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps,
    )
    print(f'  Global scheduler: {total_steps} total steps '
          f'({steps_per_stage} per stage-epoch × {MAX_EPOCHS} epochs × {NUM_STAGES} stages)')
    return scheduler


def train_one_stage(model, stage_df, optimizer, scheduler,
                    stage_num, lora_name, global_best_f1,
                    start_epoch=1, train_history=None):
    """
    Train one stage with fixed early stopping logic.

    FIX: stage_best_f1 starts at 0.0 every stage.
         Compares within-stage only, not against global best.
         This eliminates the 'adaptation dip' false-stop.
    """
    device        = next(model.parameters()).device
    train_loader  = build_stage_loader(stage_df)
    train_history = train_history or []

    # ── FIX: reset per stage, compare within stage only
    stage_best_f1      = 0.0
    stage_best_metrics = None
    patience_counter   = 0

    print()
    print('=' * 72)
    print(f'  STAGE {stage_num}/{NUM_STAGES} | {lora_name}')
    print(f'  Samples: {len(stage_df)} | Batches: {len(train_loader)} | '
          f'Max epochs: {MAX_EPOCHS} | Patience: {EARLY_STOPPING_PATIENCE} (per-stage)')
    print(f'  Global best F1 so far: {global_best_f1:.4f}')
    print('=' * 72)

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        correct = total = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, start=1):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            out  = model(input_ids=input_ids,
                         attention_mask=attention_mask, labels=labels)
            loss = out['loss'] / GRAD_ACCUM
            loss.backward()

            if (step % GRAD_ACCUM == 0) or (step == len(train_loader)):
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    MAX_GRAD_NORM,
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += out['loss'].item()
            preds   = out['logits'].argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            if step % 300 == 0 or step == len(train_loader):
                cur_lr = scheduler.get_last_lr()[0]
                print(f'   Ep{epoch} | Step {step:>4}/{len(train_loader)} | '
                      f'Loss:{total_loss/step:.4f} | Acc:{correct/total:.4f} | '
                      f'LR:{cur_lr:.2e}')

        train_acc = correct / total
        avg_loss  = total_loss / len(train_loader)
        print(f'\n  ── Ep{epoch} done | Loss={avg_loss:.4f} | '
              f'TrainAcc={train_acc:.4f}')

        val_m = evaluate(model, val_loader,
                         split_name=f'Val S{stage_num}/E{epoch}',
                         show_plots=False)
        val_m['train_acc']  = train_acc
        val_m['train_loss'] = avg_loss

        train_history.append({
            'stage': stage_num, 'epoch': epoch,
            'train_loss': avg_loss, 'train_acc': train_acc,
            'val_accuracy': val_m['accuracy'],
            'val_f1_macro': val_m['f1_macro'],
            'val_f1_weighted': val_m['f1_weighted'],
        })

        # Save every epoch
        for tag in [f'stage{stage_num}_ep{epoch}', 'LAST']:
            save_checkpoint(model, optimizer, scheduler,
                            stage_num, epoch, lora_name,
                            val_m, global_best_f1, stage_best_f1,
                            patience_counter, train_history,
                            tag=tag, is_best=False)

        # Update GLOBAL best (for final model selection)
        if val_m['f1_macro'] > global_best_f1:
            global_best_f1 = val_m['f1_macro']
            save_checkpoint(model, optimizer, scheduler,
                            stage_num, epoch, lora_name,
                            val_m, global_best_f1, stage_best_f1,
                            patience_counter, train_history,
                            tag='BEST', is_best=True)
            print(f'  🏆 New global best F1: {global_best_f1:.4f}')

        # ── FIX: per-stage early stopping
        if val_m['f1_macro'] > stage_best_f1:
            stage_best_f1      = val_m['f1_macro']
            stage_best_metrics = dict(val_m)
            patience_counter   = 0
            print(f'  ✅ Stage best F1: {stage_best_f1:.4f}')
        else:
            patience_counter += 1
            print(f'  ⏸  No stage improvement. '
                  f'patience={patience_counter}/{EARLY_STOPPING_PATIENCE}')
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f'  ⛔ Early stop: stage {stage_num}, epoch {epoch}')
                break

    # End-of-stage eval with plot
    evaluate(model, val_loader,
             split_name=f'Stage {stage_num} Final',
             show_plots=True,
             save_dir=os.path.join(CHECKPOINT_BASE, lora_name))

    return stage_best_metrics or val_m, global_best_f1, train_history


print('✅ Training loop defined')

## Cell 11 — Main Training (HPT)

Set `RUN_ALL_CONFIGS = True` to run all 3 configs sequentially.
Set `ACTIVE_CONFIG_IDX = 0` to run only `LoRA_r8` (paper best, recommended first).

In [ ]:
# ══════════════════════════════════════════════════
# SELECT WHAT TO RUN
# Recommended order: r=8 first (paper best), then r=16, r=32
# ══════════════════════════════════════════════════
RUN_ALL_CONFIGS   = False    # True → run all configs sequentially
ACTIVE_CONFIG_IDX = 1        # 0=r8 | 1=r16 | 2=r32

configs_to_run = LORA_CONFIGS if RUN_ALL_CONFIGS else [LORA_CONFIGS[ACTIVE_CONFIG_IDX]]
all_summaries  = []

for cfg in configs_to_run:
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    lora_name = cfg['name']
    print()
    print('#' * 72)
    print(f'  Experiment: {lora_name}')
    print(f'  r={cfg["r"]} | alpha={cfg["alpha"]} (={cfg["alpha"]//cfg["r"]}r) | '
          f'lr={cfg["lr"]} | dropout={cfg["dropout"]}')
    print('#' * 72)

    model    = build_model(cfg)
    trainable = [p for p in model.parameters() if p.requires_grad]

    # ── FIX: AdamW8bit (paper Table 1: 8-bit optimizer)
    optimizer = bnb.optim.AdamW8bit(
        trainable,
        lr           = cfg['lr'],
        weight_decay = WEIGHT_DECAY,
    )

    # ── FIX: Single global scheduler (not per-stage)
    scheduler = build_global_scheduler(optimizer, stages)

    all_stage_metrics = []
    train_history     = []
    best_val_f1       = 0.0

    for stage_num in range(1, NUM_STAGES + 1):
        stage_metrics, best_val_f1, train_history = train_one_stage(
            model,
            stages[stage_num - 1],
            optimizer,
            scheduler,          # same scheduler passed through all stages
            stage_num,
            lora_name,
            best_val_f1,
            train_history=train_history,
        )
        all_stage_metrics.append({'stage': stage_num, **stage_metrics})

    # Save history
    summary_dir = os.path.join(CHECKPOINT_BASE, lora_name)
    os.makedirs(summary_dir, exist_ok=True)
    with open(os.path.join(summary_dir, 'training_history.json'), 'w') as f:
        json.dump(train_history, f, indent=2, default=str)
    with open(os.path.join(summary_dir, 'training_summary.json'), 'w') as f:
        json.dump(all_stage_metrics, f, indent=2, default=str)

    all_summaries.append({
        'config':             lora_name,
        'r':                  cfg['r'],
        'alpha':              cfg['alpha'],
        'best_val_f1_macro':  best_val_f1,
        'epochs_logged':      len(train_history),
    })

    print()
    print('=' * 72)
    print(f'  TRAINING COMPLETE — {lora_name}')
    print(f'  Best Val F1-Macro : {best_val_f1:.4f}')
    print('=' * 72)

print()
print('Experiment summary:')
print(pd.DataFrame(all_summaries).to_string(index=False))

## Cell 12 — Resume from Checkpoint

In [ ]:
# ══════════════════════════════════════════════════
# Resume after Colab disconnect.
# Run Cells 1-10 first, then this cell.
# ══════════════════════════════════════════════════
RESUME_LORA_NAME = 'LoRA_r8'
RESUME_TAG       = 'LAST'   # 'LAST', 'BEST', or 'stage3_ep2'

cfg = next(c for c in LORA_CONFIGS if c['name'] == RESUME_LORA_NAME)
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

model     = build_model(cfg, adapter_path=adapter_dir(RESUME_LORA_NAME, RESUME_TAG))
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable, lr=cfg['lr'], weight_decay=WEIGHT_DECAY)
scheduler = build_global_scheduler(optimizer, stages)

state      = load_checkpoint_state(optimizer, scheduler, RESUME_LORA_NAME, tag=RESUME_TAG)
loaded_head = load_classifier_head(model, RESUME_LORA_NAME, tag=RESUME_TAG)

if state and loaded_head:
    best_val_f1   = state['best_val_f1']
    start_stage   = state['stage']
    start_epoch   = state['epoch'] + 1
    train_history = state.get('train_history', [])

    print(f'\n✅ Resuming {RESUME_LORA_NAME} from stage {start_stage}, '
          f'next epoch {start_epoch}')
    print(f'   Global best F1: {best_val_f1:.4f}')

    for stage_num in range(start_stage, NUM_STAGES + 1):
        ep = start_epoch if stage_num == start_stage else 1
        stage_metrics, best_val_f1, train_history = train_one_stage(
            model, stages[stage_num - 1],
            optimizer, scheduler,
            stage_num, RESUME_LORA_NAME, best_val_f1,
            start_epoch=ep, train_history=train_history,
        )

    print(f'\n✅ Resume complete | Final best F1: {best_val_f1:.4f}')

## Cell 13 — Final Test Evaluation

In [ ]:
RUN_TEST_FOR = 'LoRA_r8'   # change after each config finishes

cfg = next(c for c in LORA_CONFIGS if c['name'] == RUN_TEST_FOR)
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

model  = build_model(cfg, adapter_path=adapter_dir(RUN_TEST_FOR, 'BEST'),
                     is_trainable=False)
loaded = load_classifier_head(model, RUN_TEST_FOR, tag='BEST')

if loaded:
    print(f'Final test evaluation → {RUN_TEST_FOR}')
    test_dir = os.path.join(CHECKPOINT_BASE, RUN_TEST_FOR, 'final_test')
    os.makedirs(test_dir, exist_ok=True)

    test_metrics = evaluate(
        model, test_loader,
        split_name=f'TEST — {RUN_TEST_FOR}',
        show_plots=True, save_dir=test_dir,
    )
    with open(os.path.join(test_dir, 'test_metrics.json'), 'w') as f:
        json.dump({'lora_config': RUN_TEST_FOR, **test_metrics}, f,
                  indent=2, default=str)
    print(f'\n✅ Test metrics saved → {test_dir}')

## Cell 14 — HPT Comparison Table + Learning Curves

In [ ]:
# ── Summary table
rows = []
for cfg in LORA_CONFIGS:
    tf = os.path.join(CHECKPOINT_BASE, cfg['name'], 'final_test', 'test_metrics.json')
    if os.path.exists(tf):
        with open(tf) as f:
            m = json.load(f)
        rows.append({
            'Config':      cfg['name'],
            'r':           cfg['r'],
            'alpha':       cfg['alpha'],
            'alpha/r':     cfg['alpha'] // cfg['r'],
            'lr':          cfg['lr'],
            'dropout':     cfg['dropout'],
            'Test Acc':    round(m['accuracy'],    4),
            'F1 Macro':    round(m['f1_macro'],    4),
            'F1 Weighted': round(m['f1_weighted'], 4),
        })
    else:
        rows.append({'Config': cfg['name'], 'r': cfg['r'],
                     'alpha': cfg['alpha'], 'alpha/r': cfg['alpha']//cfg['r'],
                     'lr': cfg['lr'], 'dropout': cfg['dropout'],
                     'Test Acc': 'N/A', 'F1 Macro': 'N/A', 'F1 Weighted': 'N/A'})

hpt_df = pd.DataFrame(rows)
print('=' * 72)
print('  LLaMA-3.2-3B rsLoRA HPT RESULTS')
print('=' * 72)
print(hpt_df.to_string(index=False))

# ── Learning curves
completed = [c for c in LORA_CONFIGS
             if os.path.exists(os.path.join(CHECKPOINT_BASE, c['name'],
                                             'training_history.json'))]
if not completed:
    print('\nNo training history found yet.')
else:
    fig, axes = plt.subplots(1, len(completed),
                              figsize=(7 * len(completed), 5))
    if len(completed) == 1:
        axes = [axes]

    for ax, cfg in zip(axes, completed):
        with open(os.path.join(CHECKPOINT_BASE, cfg['name'],
                                'training_history.json')) as f:
            history = json.load(f)

        x      = list(range(1, len(history) + 1))
        t_loss = [h['train_loss']    for h in history]
        t_acc  = [h['train_acc']     for h in history]
        v_f1   = [h['val_f1_macro']  for h in history]
        v_acc  = [h['val_accuracy']  for h in history]

        ax.plot(x, t_loss, 'b-o',  label='Train Loss',  linewidth=2, markersize=4)
        ax.set_xlabel('Epoch (across stages)')
        ax.set_ylabel('Train Loss', color='blue')

        ax2 = ax.twinx()
        ax2.plot(x, v_f1,  'r-s',  label='Val F1 Macro', linewidth=2, markersize=4)
        ax2.plot(x, v_acc, 'g--^', label='Val Acc',       linewidth=1.5, markersize=3)
        ax2.plot(x, t_acc, 'b--',  label='Train Acc',     linewidth=1, markersize=3,
                 alpha=0.5)
        ax2.set_ylabel('Accuracy / F1')

        ax.set_title(f"{cfg['name']} | r={cfg['r']} α={cfg['alpha']}",
                     fontsize=11)
        ax.grid(alpha=0.3)

        lines1, lab1 = ax.get_legend_handles_labels()
        lines2, lab2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, lab1 + lab2, loc='best', fontsize=8)

        # Mark stage boundaries
        for s in range(1, NUM_STAGES):
            boundary = s * MAX_EPOCHS
            if boundary < len(x):
                ax.axvline(x=boundary + 0.5, color='gray',
                           linestyle=':', alpha=0.5)

    plt.suptitle('LLaMA-3.2-3B rsLoRA v2 — HPT Learning Curves\n'
                 '(vertical dotted lines = stage boundaries)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(CHECKPOINT_BASE, 'HPT_v2_comparison.png')
    plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'\n✅ HPT comparison saved → {path}')

## Cell 15 — List All Checkpoints

In [ ]:
list_checkpoints()